# Documentação das tabelas em Portugues

# Parte 1: Documentação das Tabelas

In [0]:
dbutils.widgets.text("catalog", "", "Catalog Name")
dbutils.widgets.text("schema", "", "Schema Name")

In [0]:
%pip install argostranslate langdetect

In [0]:

import argostranslate.package
import argostranslate.translate

from_code = "en"
to_code = "pt"

# Instalar o pacote de traducao Argos Translate
argostranslate.package.update_package_index()
available_packages = argostranslate.package.get_available_packages()
package_to_install = next(
    filter(
        lambda x: x.from_code == from_code and x.to_code == to_code, available_packages
    )
)
argostranslate.package.install_from_path(package_to_install.download())

In [0]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StructType, StructField, StringType
from langdetect import detect

# Funcao para trazer um descritivo para a tabela usando ai_gen
def descriptionTable(catalog, schema, table):
    tbl_schema = spark.table(f"{catalog}.{schema}.{table}").schema.simpleString()
    table_desc = spark.sql(f"""
        SELECT ai_gen("
            Human: {catalog}.{schema}.{table_name} is a single table in a data warehouse.

            Table schema: {tbl_schema}

            As an industry-leading expert Data Scientist, generate a one-paragraph summary of the provided table. This summary will be added next to the table as a description in the data explorer UI. Your description should be succinct and written in an objective and decisive tone. Start with one sentence summarizing what the table is, followed by a detailed description. Ensure the paragraph is less than 100 words. The summary should only contain English characters, commas, and periods, with each sentence being direct and straightforward. Do not include quotation marks. Avoid flowery language, descriptions of the schema itself, decorative tone, specific examples, parentheses, and single or double quotes in your answer. Do not show table name or schema name or catalog name.
        ") as table_description;
    """).collect()[0][0]

    return table_desc

# Funcao para traduzir a descricao
def descriptionTranslate(table_desc_orig):
    table_desc = argostranslate.translate.translate(table_desc_orig, from_code, to_code)
    return table_desc
  
# Funcao para atualizar a descricao das tabelas
def updateTable(catalog, schema, table_name, new_description):
  new_description = new_description.replace("'", "")
  update_description = f"ALTER TABLE {catalog}.{schema}.{table_name} SET TBLPROPERTIES ('comment' = '{new_description}')"
  spark.sql(update_description)  

# Funcao para verificar o idioma do comentario ja existente
def descriptionCheck(actual_description):
  lang = detect(actual_description)
  return lang

In [0]:
def descriptionColumn(catalog, schema, table, col):
  return spark.sql(f"""
    SELECT ai_gen("Human: {col} is a column in the table {catalog}.{schema}.{table}. 

    As an industry-leading expert Data Scientist, generate a concise description for this column. Start with one sentence summarizing what the column represents, followed by more details about its significance or role in the table. Keep the description under 25 words. Ensure the paragraph is in English, with each sentence being clear and precise. Avoid mentioning the table name, column name, column type, or using parentheses, quotation marks, or examples.") as column_description;
    """).collect()[0][0]

def updateColumn(catalog, schema, table, col, comment):
  spark.sql(f"ALTER TABLE {catalog}.{schema}.{table} ALTER COLUMN {col} COMMENT '{comment}'")

def descriptionColumns(catalog, schema, table):
    
  # busca as colunas da tabela e seus comentários
  columns = spark.sql(f"describe table {catalog}.{schema}.{table}").collect()

  # para cada coluna
  for column in columns:

    # se não houver descrição
    if column.comment is None or column.comment == "":
      # gera a descrição para cada coluna
      desc = descriptionColumn(catalog, schema, table, column.col_name)
      # traduz o comentario com ArgosTranslate
      translated_desc = descriptionTranslate(desc)
    
    # se houver descrição, mas não traduzida
    elif detect(column.comment) == from_code:
      # traduz o comentario com ArgosTranslate
      translated_desc = descriptionTranslate(column.comment)
    
    # se já tiver comentario traduzido, ignora
    else:
      translated_desc = None
      print(f'Column {catalog}.{schema}.{table}.{column.col_name} already has description.')
    
    # se foi gerada uma nova descrição
    if translated_desc:
      # atualiza o comentário da coluna
      try:
        updateColumn(catalog, schema, table, column.col_name, translated_desc)
      except:
        print(f"Error updating column {catalog}.{schema}.{table}.{column.col_name}.")

In [0]:
# Listar todas as tabelas do catalogo.schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

tables = spark.catalog.listTables(f"{catalog}.{schema}")
table_names = [table.name for table in tables if table.name != "_sqldf"]

for table_name in table_names:

    if spark.catalog.tableExists(f"{catalog}.{schema}.{table_name}"):
        print("========")
        print(f"{catalog}.{schema}.{table_name}")
        print("---")
        print("Generating table description...")

        # verificar se a tabela já tem comentario
        desc = spark.sql(f"DESCRIBE DETAIL {catalog}.{schema}.{table_name}").collect()[0]["description"]
        
        # se nao tiver comentario ainda
        if desc is None or desc == "":
            # gera a descrição
            gen_desc = descriptionTable(catalog, schema, table_name)
            # traduz o comentario com ArgosTranslate
            translated_desc = descriptionTranslate(gen_desc)

        # se ja tiver comentario, mas não traduzido
        elif detect(desc) == from_code:
            # traduz o comentario com ArgosTranslate
            translated_desc = descriptionTranslate(desc)
        
        # se já tiver comentario traduzido, ignora
        else:
            translated_desc = None
            print('Table already has description.')

        # se foi gerada uma nova descrição
        if translated_desc:
            print(translated_desc)
            # atualiza a descrição da tabela
            try:
                updateTable(catalog, schema, table_name, translated_desc)
            except:
                print(f"Error updating table {catalog}.{schema}.{table}.")

        # gera descrições para as colunas
        print("---")
        print("Generating column descriptions...")
        descriptionColumns(catalog, schema, table_name)

    else:
        print("========")
        print(f"Table {table_name} does not exist.")

# Parte 2: Chatbot do Catálogo de Dados

In [0]:
%sql
create or replace table rodrigo_catalog.fsi_credit_decisioning.descriptions as
select table_catalog, table_schema, table_name, comment
  from system.information_schema.tables
 where table_catalog = 'rodrigo_catalog'
   and table_schema = 'fsi_credit_decisioning'

In [0]:
%sql
ALTER TABLE rodrigo_catalog.fsi_credit_decisioning.descriptions SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
%pip install mlflow==2.9.0 lxml==4.9.3 transformers==4.30.2 langchain==0.0.344 databricks-vectorsearch==0.22
%pip install databricks-sdk dbtunnel[gradio] gradio
dbutils.library.restartPython()

In [0]:
import mlflow.deployments
deploy_client = mlflow.deployments.get_deploy_client("databricks")

In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient()

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

vs_index_fullname = f"{catalog}.{schema}.descriptions_index"
source_table_fullname = f"{catalog}.{schema}.descriptions"
VECTOR_SEARCH_ENDPOINT_NAME = "one-env-shared-endpoint-4"

vsc.create_delta_sync_index(
  endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
  index_name=vs_index_fullname,
  source_table_name=source_table_fullname,
  pipeline_type="TRIGGERED",
  primary_key="table_name",
  embedding_source_column='comment',
  embedding_model_endpoint_name='databricks-bge-large-en'
)

In [0]:
from langchain.chains import RetrievalQA, RetrievalQAWithSourcesChain
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatDatabricks
from langchain.vectorstores import DatabricksVectorSearch
from langchain.embeddings import DatabricksEmbeddings
import os

host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
os.environ['DATABRICKS_TOKEN'] = dbutils.secrets.get("rodrigo_llm", "teste123")

def get_retriever(persist_dir: str = None):
    embedding_model = DatabricksEmbeddings(endpoint="databricks-bge-large-en")
    os.environ["DATABRICKS_HOST"] = host
    
    vsc = VectorSearchClient(workspace_url=host, personal_access_token=os.environ["DATABRICKS_TOKEN"])
    vs_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=vs_index_fullname
    )

    # Create the retriever
    vectorstore = DatabricksVectorSearch(
        vs_index, text_column="comment", embedding=embedding_model
    )
    return vectorstore.as_retriever()



In [0]:
chat_model = ChatDatabricks(endpoint="databricks-meta-llama-3-70b-instruct", max_tokens = 500)

TEMPLATE = """Você é um assistente de catalogo de dados. Você ajuda os usuários da Databricks a realizar pesquisas de metadados.
Use as seguintes partes do contexto para responder à pergunta no final.
Responda em Português:
{context}
Pergunta: {question}
Resposta: 
"""
prompt = PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])

chain = RetrievalQA.from_chain_type(
    llm=chat_model,
    chain_type="stuff",
    retriever=get_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

In [0]:
question = "Me traga todas as tabelas que tenham relacao com precos dos produtos"

json = chain.invoke({"query": question})['result']
print(json)